In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, applications
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import classification_report
import numpy as np
import matplotlib.pyplot as plt

# Data parameters
IMG_SIZE = 256
BATCH_SIZE = 32

# Data augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='reflect',
    validation_split=0.2
)

# Data generators
train_generator = train_datagen.flow_from_directory(
    '/kaggle/input/apple-tree-leaf-disease-dataset',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42
)

val_generator = train_datagen.flow_from_directory(
    '/kaggle/input/apple-tree-leaf-disease-dataset',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

# Load pre-trained Xception model
base_model = applications.Xception(
    include_top=False,
    weights='imagenet',
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    pooling='avg'
)

# Freeze base model
base_model.trainable = False

# Build model
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.Dropout(0.5)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.BatchNormalization()(x)
outputs = layers.Dense(5, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

# Initial training with fixed learning rate
initial_learning_rate = 0.001
model.compile(
    optimizer=optimizers.Adam(learning_rate=initial_learning_rate),
    loss='categorical_crossentropy',
    metrics=[tf.keras.metrics.CategoricalAccuracy(name='accuracy')]  # Explicit metric definition
)

# Callbacks for initial training
initial_callbacks = [
    EarlyStopping(patience=10, monitor='val_accuracy', restore_best_weights=True),
    ModelCheckpoint('initial_model.h5', monitor='val_accuracy', save_best_only=True),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=5,
        min_lr=1e-6,
        verbose=1
    )
]

# Initial training
print("Initial training phase...")
history = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    validation_data=val_generator,
    validation_steps=len(val_generator),
    epochs=30,
    callbacks=initial_callbacks,
    verbose=1
)

# Fine-tuning
print("\nFine-tuning phase...")
base_model.trainable = True
for layer in base_model.layers[:50]:
    layer.trainable = False

# Recompile with lower learning rate
fine_tune_lr = 0.00001
model.compile(
    optimizer=optimizers.Adam(learning_rate=fine_tune_lr),
    loss='categorical_crossentropy',
    metrics=[tf.keras.metrics.CategoricalAccuracy(name='accuracy')]  # Consistent metric definition
)

# Callbacks for fine-tuning
fine_tune_callbacks = [
    EarlyStopping(patience=10, monitor='val_accuracy', restore_best_weights=True),
    ModelCheckpoint('fine_tuned_model.h5', monitor='val_accuracy', save_best_only=True)
]

history_fine = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    validation_data=val_generator,
    validation_steps=len(val_generator),
    epochs=20,
    initial_epoch=history.epoch[-1] + 1,
    callbacks=fine_tune_callbacks,
    verbose=1
)

# Evaluation
print("\nEvaluating model...")
val_pred = model.predict(val_generator, steps=len(val_generator))
val_pred_classes = np.argmax(val_pred, axis=1)

print("\nClassification Report:")
print(classification_report(
    val_generator.classes,
    val_pred_classes,
    target_names=list(train_generator.class_indices.keys())
))

# Save final model
model.save('final_apple_leaf_model.h5')
print("\nModel saved as 'final_apple_leaf_model.h5'")

2025-08-02 16:48:19.020103: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754153299.192136      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754153299.240292      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Found 1315 images belonging to 5 classes.
Found 326 images belonging to 5 classes.


I0000 00:00:1754153315.106532      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Initial training phase...


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30


I0000 00:00:1754153326.639696      74 service.cc:148] XLA service 0x7fc0dc003ad0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1754153326.640472      74 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1754153327.696513      74 cuda_dnn.cc:529] Loaded cuDNN version 90300


 2/42 ━━━━━━━━━━━━━━━━━━━━ 3s 90ms/step - accuracy: 0.3203 - loss: 1.8679 

I0000 00:00:1754153333.609685      74 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


42/42 ━━━━━━━━━━━━━━━━━━━━ 55s 1s/step - accuracy: 0.5972 - loss: 1.1374 - val_accuracy: 0.8160 - val_loss: 0.7269 - learning_rate: 0.0010
Epoch 2/30
42/42 ━━━━━━━━━━━━━━━━━━━━ 25s 604ms/step - accuracy: 0.8011 - loss: 0.5655 - val_accuracy: 0.7975 - val_loss: 0.6100 - learning_rate: 0.0010
Epoch 3/30
42/42 ━━━━━━━━━━━━━━━━━━━━ 25s 601ms/step - accuracy: 0.8632 - loss: 0.4120 - val_accuracy: 0.8067 - val_loss: 0.5405 - learning_rate: 0.0010
Epoch 4/30
42/42 ━━━━━━━━━━━━━━━━━━━━ 26s 612ms/step - accuracy: 0.8745 - loss: 0.3789 - val_accuracy: 0.8742 - val_loss: 0.4218 - learning_rate: 0.0010
Epoch 5/30
42/42 ━━━━━━━━━━━━━━━━━━━━ 25s 602ms/step - accuracy: 0.8787 - loss: 0.3415 - val_accuracy: 0.8313 - val_loss: 0.4735 - learning_rate: 0.0010
Epoch 6/30
42/42 ━━━━━━━━━━━━━━━━━━━━ 26s 608ms/step - accuracy: 0.8865 - loss: 0.3129 - val_accuracy: 0.8589 - val_loss: 0.3991 - learning_rate: 0.0010
Epoch 7/30
42/42 ━━━━━━━━━━━━━━━━━━━━ 26s 609ms/step - accuracy: 0.9142 - loss: 0.2758 - val_acc